# New learners on their own — NCHS perinatal model, version 2

Framework stage: **S8** (preliminary run). Spanish version: `08_new_learners_es.ipynb` (same code).

**Aim.** Test whether a learner different from those used so far discriminates better than XGBoost with the same data of version 2: 22 weeks or more, 28 parameters, all records and median imputation. Each learner is assessed **on its own**, with no ensemble.

**New learners:**
- **Logistic regression with splines**: non-linear relations in the continuous predictors.
- **EBM** (*Explainable Boosting Machine*): interpretable additive model with pairwise interactions.
- **MLP neural network** (multilayer perceptron).
- **CatBoost**: another form of boosting.

**Same rules as before:**
- Same sample and same 80/20 split as notebook 07.
- 20 random hyperparameter combinations per learner, with 3 inner folds on 30% of the fit part.
- Reference XGBoost with its hyperparameters from notebook 07.
- **Rule fixed before running:** a new learner replaces XGBoost as primary only if its AUROC is at least 0.005 higher and the 95% CI of the difference is above 0. If one does, its formal evaluation and gate G4 follow. If none does, the classification of version 2 stands.
- The correlation between the learners' predictions is also reported. It will be used later to decide whether an ensemble is worth trying.

**Requirements:** `%pip install catboost interpret` (plus `scikit-learn` 1.7 or later and `xgboost`). Needs the version 2 contract table (notebook 07). Estimated time: 2 to 3 hours, of which the EBM is the longest part. Output: `data/outputs_08_new_learners.txt`.

In [ ]:
import sys, os, time, json
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.optimize import brentq
from sklearn.base import clone
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import SplineTransformer
from sklearn.neural_network import MLPClassifier
from catboost import CatBoostClassifier                                   # %pip install catboost
from interpret.glassbox import ExplainableBoostingClassifier              # %pip install interpret

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
INTERIM_DIR = Path(os.environ.get("NCHS_INTERIM_DIR", ROOT / "data" / "interim"))
CONTRACT_DIR = Path(os.environ.get("NCHS_CONTRACT3_DIR", ROOT / "data" / "contract" / "perinatal_nchs_v2"))
OUT_FILE = ROOT / "data" / "outputs_08_new_learners.txt"
SEED = 20261121
YEARS, DEV_YEARS, TEST_YEARS = list(range(2016, 2024)), list(range(2016, 2022)), [2022, 2023]
FRACTION = float(os.environ.get("NCHS_FRACTION", 0.02))
QUICK = os.environ.get("NCHS_QUICK") == "1"
N_JOBS = int(os.environ.get("NCHS_JOBS", -1))
lines = []
def show(title, obj):
    txt = obj.to_string() if hasattr(obj, "to_string") else str(obj)
    block = f"\n== {title}\n{txt}"
    print(block); lines.append(block)

def coded(s, unknown):
    s = s.astype("Float64").astype(float)
    return s.where(~s.isin(unknown))

def yn(s):
    s = s.astype("string")
    y = s.eq("Y").fillna(False).to_numpy(bool); n = s.eq("N").fillna(False).to_numpy(bool)
    return pd.Series(np.where(y, 1.0, np.where(n, 0.0, np.nan)), index=s.index)

V1_DIR = Path(os.environ.get("NCHS_CONTRACT_DIR", ROOT / "data" / "contract" / "perinatal_nchs_v1"))
NEW = ["gestational_diabetes", "gestational_hypertension", "eclampsia", "smoked_during_pregnancy"]

import sklearn
assert tuple(int(v) for v in sklearn.__version__.split(".")[:2]) >= (1, 7), (
    f"scikit-learn {sklearn.__version__}: the MLP needs 1.7 or later to accept sample weights. "
    "Run: %pip install -U scikit-learn>=1.7 (then restart the kernel).")


## 1. Data and split (same as notebook 07)

In [ ]:
# Same data and same split as notebook 07 (version 2: 22 weeks or more, 28 parameters, all records),
# so the new learners are compared with XGBoost on exactly the same comparison part.
DROP = ["unit_id", "group_id", "period", "component", "ga_weeks", "label_main", "label_fetal",
        "label_early_neonatal", "estofd", "label_ge28", "n_unknown"]
PREDICTORS = [c for c in pd.read_parquet(CONTRACT_DIR / "year_2023.parquet").columns if c not in DROP]
EXTRA = ["label_main", "label_fetal", "label_early_neonatal", "ga_weeks", "estofd", "n_unknown"]
def load(years, seed):
    rng = np.random.default_rng(seed); parts = []; n = e = 0
    for y in years:
        c = pd.read_parquet(CONTRACT_DIR / f"year_{y}.parquet", columns=PREDICTORS + EXTRA)
        n += len(c); e += int(c["label_main"].sum())
        parts.append(c[c["label_main"].to_numpy() | (rng.random(len(c)) < FRACTION)])
    d = pd.concat(parts, ignore_index=True)
    if QUICK:
        d = d.sample(n=min(len(d), 40000), random_state=seed).reset_index(drop=True)
    d["w"] = np.where(d["label_main"], 1.0, 1.0 / FRACTION)
    return d, n, e

dev, n_dev, e_dev = load(DEV_YEARS, SEED)
X = dev[PREDICTORS].astype("float32"); y = dev["label_main"].to_numpy(bool); w = dev["w"].to_numpy()
X_tr, X_te, y_tr, y_te, w_tr, w_te = train_test_split(X, y, w, test_size=0.2, stratify=y, random_state=SEED)
CONTINUOUS = [c for c in PREDICTORS if X_tr[c].nunique() > 2]
show("Data (same sample and split as notebook 07)", pd.Series({
    "development products (2016-2021)": n_dev, "development events": e_dev, "sample rows": len(dev),
    "fit/tune part": len(X_tr), "comparison part": len(X_te), "events in comparison part": int(y_te.sum()),
    "continuous predictors (splines)": ", ".join(CONTINUOUS)}))

## 2. Random search of each new learner

In [ ]:
# New learners, each with the SAME search budget as before: N_CONFIGS combinations drawn from
# prespecified ranges, AUROC in 3 inner folds of a stratified 30% subsample of the fit part, best
# combination refitted on the whole fit part. All of them impute the median inside training (D3), so
# none can read the certificate type from the unknown codes. Reference: XGBoost of notebook 07 with
# its tuned hyperparameters (refitted here on the same fit part, not searched again).
N_CONFIGS = 3 if QUICK else 20
SEARCH_SHARE = 0.30
rs = np.random.default_rng(SEED + 8)
loguni = lambda lo, hi: float(np.exp(rs.uniform(np.log(lo), np.log(hi))))
XGB_V2 = dict(max_depth=7, learning_rate=0.0105, min_child_weight=79.6815, subsample=0.9275,
              colsample_bytree=0.6472, reg_lambda=5.5315, reg_alpha=0.0429)

def sample_config(name):
    if name == "LR splines":
        return {"n_knots": int(rs.choice([4, 5, 6, 8])), "C": loguni(1e-4, 10)}
    if name == "EBM":
        return {"interactions": int(rs.choice([0, 5, 10, 20])), "learning_rate": loguni(0.005, 0.05),
                "min_samples_leaf": int(loguni(2, 200)), "max_bins": int(rs.choice([128, 256, 512]))}
    if name == "MLP":
        return {"hidden_layer_sizes": [(64,), (128,), (64, 32), (128, 64), (256, 128)][rs.integers(5)],
                "alpha": loguni(1e-5, 1e-1), "learning_rate_init": loguni(1e-4, 1e-2),
                "batch_size": int(rs.choice([256, 512, 1024]))}
    return {"depth": int(rs.integers(4, 11)), "learning_rate": loguni(0.01, 0.2), "l2_leaf_reg": loguni(1, 30),
            "random_strength": loguni(0.1, 10), "bagging_temperature": float(rs.uniform(0, 1))}

def build(name, cfg, n_estimators=None):
    imp = SimpleImputer(strategy="median")
    if name == "LR splines":
        cfg = dict(cfg); k = cfg.pop("n_knots")
        prep = ColumnTransformer([("spl", SplineTransformer(n_knots=k, degree=3), [PREDICTORS.index(c) for c in CONTINUOUS])], remainder="passthrough")
        return make_pipeline(imp, prep, StandardScaler(), LogisticRegression(solver="liblinear", max_iter=1000, **cfg))
    if name == "EBM":
        return make_pipeline(imp, ExplainableBoostingClassifier(outer_bags=8, n_jobs=N_JOBS, random_state=SEED, **cfg))
    if name == "MLP":
        return make_pipeline(imp, StandardScaler(), MLPClassifier(early_stopping=True, max_iter=200, random_state=SEED, **cfg))
    if name == "CatBoost":
        return make_pipeline(imp, CatBoostClassifier(iterations=n_estimators or (100 if QUICK else 2000), verbose=0,
                                                     thread_count=N_JOBS, random_seed=SEED, **cfg))
    return make_pipeline(imp, XGBClassifier(n_estimators=n_estimators, tree_method="hist", n_jobs=N_JOBS,
                                            random_state=SEED, eval_metric="auc", **cfg))

def fit_w(est, X_, y_, w_):
    return est.fit(X_, y_, **{f"{est.steps[-1][0]}__sample_weight": w_})

X_se, _, y_se, _, w_se, _ = train_test_split(X_tr, y_tr, w_tr, train_size=SEARCH_SHARE, stratify=y_tr, random_state=SEED)
inner = list(StratifiedKFold(3, shuffle=True, random_state=SEED).split(X_se, y_se))
NEW_LEARNERS = ["LR splines", "EBM", "MLP", "CatBoost"]
fitted, pred, grid_rows, trials = {}, {}, [], []
t = time.time()
fitted["XGB"] = fit_w(build("XGB", XGB_V2, n_estimators=50 if QUICK else 832), X_tr, y_tr, w_tr)
pred["XGB"] = fitted["XGB"].predict_proba(X_te)[:, 1]
print("XGB (reference) done in", round((time.time() - t) / 60, 1), "min")
for name in NEW_LEARNERS:
    t = time.time(); best = (-1, None, None)
    for k in range(N_CONFIGS):
        cfg = sample_config(name); aucs, iters = [], []
        for tr, va in inner:
            if name == "CatBoost":
                imp = SimpleImputer(strategy="median").fit(X_se.iloc[tr])
                cb = build(name, cfg).steps[-1][1]
                cb.fit(imp.transform(X_se.iloc[tr]), y_se[tr], sample_weight=w_se[tr],
                       eval_set=(imp.transform(X_se.iloc[va]), y_se[va]), early_stopping_rounds=50, use_best_model=True)
                p = cb.predict_proba(imp.transform(X_se.iloc[va]))[:, 1]; iters.append(cb.get_best_iteration() + 1)
            else:
                p = fit_w(build(name, cfg), X_se.iloc[tr], y_se[tr], w_se[tr]).predict_proba(X_se.iloc[va])[:, 1]
            aucs.append(roc_auc_score(y_se[va], p))
        score = float(np.mean(aucs))
        trials.append({"learner": name, "inner AUROC": score, **{k_: (round(v, 4) if isinstance(v, float) else str(v)) for k_, v in cfg.items()}})
        if score > best[0]:
            best = (score, cfg, int(np.mean(iters)) if iters else None)
    score, cfg, n_iter = best
    fitted[name] = fit_w(build(name, cfg, n_estimators=n_iter), X_tr, y_tr, w_tr)
    pred[name] = fitted[name].predict_proba(X_te)[:, 1]
    shown = {k_: (round(v, 4) if isinstance(v, float) else v) for k_, v in cfg.items()}
    if n_iter: shown["iterations"] = n_iter
    grid_rows.append({"learner": name, "best combination": json.dumps(shown, default=str), "inner AUROC": round(score, 4),
                      "minutes": round((time.time() - t) / 60, 1)})
    print(name, "done in", grid_rows[-1]["minutes"], "min")
show(f"Random search of the new learners ({N_CONFIGS} combinations each, 3 inner folds on {int(SEARCH_SHARE * 100)}% of the fit part)",
     pd.DataFrame(grid_rows).set_index("learner"))
show("Spread of the inner AUROC over the combinations tried",
     pd.DataFrame(trials).groupby("learner")["inner AUROC"].agg(["min", "median", "max"]).round(4))
trials_df = pd.DataFrame(trials)
for name in NEW_LEARNERS:
    t_ = trials_df[trials_df["learner"] == name].dropna(axis=1, how="all").drop(columns="learner")
    show(f"All combinations tried: {name}", t_.sort_values("inner AUROC", ascending=False).round(4).reset_index(drop=True))

## 3. Comparison on their own, selection rule, correlation between learners and confusion matrices

In [ ]:
# Comparison on the same 20% part as notebook 07, each learner on its own (no ensemble).
B = 50 if QUICK else 300
boot = np.random.default_rng(SEED)
idx_pos, idx_neg = np.where(y_te)[0], np.where(~y_te)[0]
boot_idx = [np.concatenate([boot.choice(idx_pos, len(idx_pos)), boot.choice(idx_neg, len(idx_neg))]) for _ in range(B)]
def wcut(p, share):
    o = np.argsort(p); cw = np.cumsum(w_te[o])
    return p[o][np.searchsorted(cw, (1 - share) * cw[-1])]
rows = {}
for name, p in pred.items():
    a = [roc_auc_score(y_te[ix], p[ix]) for ix in boot_idx]
    d = [roc_auc_score(y_te[ix], p[ix]) - roc_auc_score(y_te[ix], pred["XGB"][ix]) for ix in boot_idx]
    top = p >= wcut(p, 0.10)
    rows[name] = {"AUROC": round(roc_auc_score(y_te, p), 4), "AUROC 2.5%": round(np.percentile(a, 2.5), 4),
                  "AUROC 97.5%": round(np.percentile(a, 97.5), 4),
                  "minus XGB": round(roc_auc_score(y_te, p) - roc_auc_score(y_te, pred["XGB"]), 4),
                  "minus XGB 2.5%": round(np.percentile(d, 2.5), 4), "minus XGB 97.5%": round(np.percentile(d, 97.5), 4),
                  "AUPRC (weighted)": round(average_precision_score(y_te, p, sample_weight=w_te), 4),
                  "observed/expected": round(float(y_te.sum() / np.sum(w_te * p)), 3),
                  "events in top 10% (%)": round(100 * float(y_te[top].sum() / y_te.sum()), 1)}
res = pd.DataFrame(rows).T.sort_values("AUROC", ascending=False)
show(f"Each learner on its own, comparison part (prevalence {y_te.sum() / w_te.sum():.5f})", res)

# Prespecified rule: a new learner replaces XGBoost as primary only if its AUROC is at least 0.005 higher
# and the 95% CI of the difference is above 0 (same tolerance as the selection rule of notebook 04).
winners = [k for k in NEW_LEARNERS if res.loc[k, "minus XGB"] >= 0.005 and res.loc[k, "minus XGB 2.5%"] > 0]
primary = max(winners, key=lambda k: res.loc[k, "AUROC"]) if winners else "XGB"
show("Selection", pd.Series({"best new learner": max(NEW_LEARNERS, key=lambda k: res.loc[k, "AUROC"]),
                             "learners that beat XGBoost by 0.005 or more (CI > 0)": ", ".join(winners) or "none",
                             "primary learner": primary,
                             "next step": "formal evaluation of the new primary learner" if winners else
                                          "primary unchanged: the classification of version 2 stands"}))

# Diversity: correlation of the predictions (on the logit scale) and of the risk ranking. Low correlation
# is what an ensemble would need; it is reported for a later decision, no ensemble is fitted here.
lg = pd.DataFrame({k: np.log(np.clip(p, 1e-7, 1 - 1e-7) / (1 - np.clip(p, 1e-7, 1 - 1e-7))) for k, p in pred.items()})
show("Correlation of the predictions between learners (Spearman, comparison part)", lg.corr(method="spearman").round(3))

cm = []
for name, p in pred.items():
    flag = p >= wcut(p, 0.10)
    tp, fn = w_te[flag & y_te].sum(), w_te[~flag & y_te].sum(); fp, tn = w_te[flag & ~y_te].sum(), w_te[~flag & ~y_te].sum()
    cm.append({"learner": name, "TP": round(tp), "FN": round(fn), "FP": round(fp), "TN": round(tn),
               "accuracy": round((tp + tn) / (tp + tn + fp + fn), 4), "sensitivity": round(tp / (tp + fn), 3),
               "specificity": round(tn / (tn + fp), 4), "PPV": round(tp / (tp + fp), 4), "NPV": round(tn / (tn + fn), 5)})
show("Confusion matrices at the top 10% of risk (weighted to the population of the comparison part)", pd.DataFrame(cm).set_index("learner"))
OUT_FILE.write_text("New learners on their own (perinatal model, version 2)\n" + "\n".join(lines) + "\n", encoding="utf-8")